# NeuroTrace — Entraînement du modèle IA (Colab)

Ce notebook entraîne un modèle qui reconnaît 4 classes d'IRM cérébrales : **glioma, meningioma, notumor, pituitary**.

**Comment l'utiliser**
- Exécute les cellules **une par une, dans l'ordre**, avec `Shift + Entrée`.
- Attends qu'une cellule soit terminée (le rond à gauche s'arrête) avant de lancer la suivante.
- Si tu fermes Colab ou si la session expire, il faut tout relancer depuis le début.

**Durée totale** : environ 15 à 20 minutes avec un GPU.

*Prototype pédagogique : ce modèle ne sert pas à établir un diagnostic médical.*

## Étape 0 — Vérifier que le GPU est actif

Un GPU rend l'entraînement beaucoup plus rapide. Si la cellule affiche « PAS de GPU », va dans le menu
**Exécution → Modifier le type d'exécution → T4 GPU → Enregistrer**, puis relance la cellule.

In [ ]:
import torch

print("PyTorch :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("PAS de GPU : menu Exécution > Modifier le type d'exécution > T4 GPU")

## Étape 1 — Télécharger le dataset

Le dataset est **Brain Tumor MRI Dataset** (Kaggle, environ 7 000 images, 4 classes).
Le téléchargement se fait directement dans Colab, rien à envoyer depuis ton ordinateur.

Si Kaggle demande de se connecter, colle ton **token API** quand la cellule te le demande.

In [ ]:
!pip install -q kagglehub

In [ ]:
import kagglehub

HANDLE = "masoudnickparvar/brain-tumor-mri-dataset"

try:
    dataset_path = kagglehub.dataset_download(HANDLE)
except Exception as e:
    print("Téléchargement sans connexion impossible :", type(e).__name__)
    print("Connexion à Kaggle : colle ton token quand il est demandé.")
    kagglehub.login()
    dataset_path = kagglehub.dataset_download(HANDLE)

print("Dataset téléchargé ici :", dataset_path)

## Étape 2 — Explorer le dataset

On repère les dossiers d'entraînement (`Training`) et de test (`Testing`), puis on compte les images par classe.
**Il est important de connaître ses données** : si une classe avait très peu d'images, le modèle l'apprendrait mal.

In [ ]:
from pathlib import Path

root = Path(dataset_path)

def find_split(root, names):
    for p in root.rglob("*"):
        if p.is_dir() and p.name.lower() in names:
            return p
    return None

train_dir = find_split(root, {"training", "train"})
test_dir = find_split(root, {"testing", "test"})
assert train_dir and test_dir, "Dossiers Training/Testing introuvables"

IMG_EXT = {".jpg", ".jpeg", ".png"}

def count_images(split_dir):
    counts = {}
    for c in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        counts[c.name] = sum(1 for f in c.iterdir() if f.suffix.lower() in IMG_EXT)
    return counts

print("Dossier d'entraînement :", train_dir)
print("Dossier de test        :", test_dir)
print()
train_counts, test_counts = count_images(train_dir), count_images(test_dir)
print(f"{'classe':<12}{'train':>8}{'test':>8}")
for c in train_counts:
    print(f"{c:<12}{train_counts[c]:>8}{test_counts.get(c, 0):>8}")
print(f"{'TOTAL':<12}{sum(train_counts.values()):>8}{sum(test_counts.values()):>8}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

classes = sorted(train_counts)
fig, axes = plt.subplots(len(classes), 3, figsize=(8, 2.7 * len(classes)))
for r, c in enumerate(classes):
    files = sorted(f for f in (train_dir / c).iterdir() if f.suffix.lower() in IMG_EXT)[:3]
    for k, f in enumerate(files):
        axes[r][k].imshow(Image.open(f).convert("RGB"))
        axes[r][k].axis("off")
        if k == 0:
            axes[r][k].set_title(c, loc="left")
plt.tight_layout()
plt.show()

## Étape 3 — Préparer les données

Trois groupes d'images, qui ne se mélangent jamais :
- **Train** (85 % du dossier Training) : le modèle apprend dessus.
- **Validation** (15 % du dossier Training) : sert à choisir le meilleur moment de l'entraînement.
- **Test** (dossier Testing) : **jamais vu pendant l'entraînement**, sert uniquement à la note finale.

Chaque image est redimensionnée en 224 × 224 et normalisée comme l'attend le modèle pré-entraîné.
Pour l'entraînement, on ajoute de petites variations aléatoires (légère rotation, miroir, contraste). C'est l'**augmentation de données** : elle aide le modèle à mieux généraliser.

In [ ]:
import random
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # valeurs standard d'ImageNet
BATCH_SIZE = 32

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

full_train_aug = datasets.ImageFolder(train_dir, transform=train_tf)
full_train_eval = datasets.ImageFolder(train_dir, transform=eval_tf)
test_ds = datasets.ImageFolder(test_dir, transform=eval_tf)

class_names = full_train_aug.classes
assert class_names == test_ds.classes, "Les classes de train et de test sont différentes"

idx = np.arange(len(full_train_aug))
train_idx, val_idx = train_test_split(
    idx, test_size=0.15, stratify=full_train_aug.targets, random_state=SEED)

train_ds = Subset(full_train_aug, train_idx)
val_ds = Subset(full_train_eval, val_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("Classes :", class_names)
print(f"Train : {len(train_ds)} | Validation : {len(val_ds)} | Test : {len(test_ds)}")

## Étape 4 — Construire le modèle (transfer learning)

On part de **ResNet18**, un réseau déjà entraîné sur plus d'un million d'images quotidiennes (chats, voitures, etc.).
Il sait déjà reconnaître des contours, des textures et des formes. On remplace seulement sa dernière couche pour qu'elle
donne **4 réponses** (nos 4 classes) au lieu de 1000. C'est le *transfer learning* : beaucoup plus rapide et plus précis
que d'entraîner un réseau à partir de zéro avec peu d'images.

*Pourquoi ResNet18 ?* Il est petit (rapide à entraîner et à utiliser sur un ordinateur sans GPU), très documenté, et il
fonctionne bien avec Grad-CAM, que nous ajouterons ensuite.

In [ ]:
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def build_model(n_classes, pretrained=True):
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    m = models.resnet18(weights=weights)
    m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(m.fc.in_features, n_classes))
    return m

model = build_model(len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

n_params = sum(p.numel() for p in model.parameters())
print(f"Modèle : ResNet18 — {n_params/1e6:.1f} millions de paramètres — appareil : {device}")

## Étape 5 — Entraîner

L'entraînement se fait en **deux phases** :
1. **Phase 1 (3 époques)** : on ne modifie que la dernière couche (le reste est gelé). Le modèle apprend à utiliser ce qu'il sait déjà pour nos 4 classes.
2. **Phase 2 (8 époques)** : on dégèle tout le réseau et on l'ajuste finement, avec un pas d'apprentissage plus petit.

Une **époque** = le modèle a vu une fois toutes les images d'entraînement. À chaque époque, on affiche la précision
sur l'entraînement et sur la validation, et on sauvegarde automatiquement le meilleur modèle (`best.pt`).

**Ce que tu dois voir** : la précision de validation monte et dépasse en général 95 %. Si elle reste bloquée autour de 25 %,
il y a un problème : préviens-moi.

In [ ]:
import time

def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
                out = model(x)
                loss = criterion(out, y)
            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc = 0.0

def fit(epochs, lr, phase_name):
    global best_val_acc
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    print(f"--- {phase_name} ---")
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = run_epoch(model, train_loader, optimizer)
        vl, va = run_epoch(model, val_loader)
        scheduler.step()
        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_acc"].append(ta); history["val_acc"].append(va)
        saved = ""
        if va > best_val_acc:
            best_val_acc = va
            torch.save(model.state_dict(), "best.pt")
            saved = "  <- meilleur modèle sauvegardé"
        print(f"Époque {epoch}/{epochs} | train {ta:.3f} | val {va:.3f} | {time.time()-t0:.0f}s{saved}")

In [ ]:
# Phase 1 : on gèle tout sauf la dernière couche
PHASE1_EPOCHS, LR1 = 3, 1e-3
for p in model.parameters():
    p.requires_grad = False
for p in model.fc.parameters():
    p.requires_grad = True

fit(PHASE1_EPOCHS, LR1, "Phase 1 : dernière couche seulement")

In [ ]:
# Phase 2 : on dégèle tout le réseau (ajustement fin)
PHASE2_EPOCHS, LR2 = 8, 1e-4
for p in model.parameters():
    p.requires_grad = True

fit(PHASE2_EPOCHS, LR2, "Phase 2 : ajustement fin de tout le réseau")
print(f"\nMeilleure précision de validation : {best_val_acc:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
epochs_axis = range(1, len(history["train_acc"]) + 1)
ax[0].plot(epochs_axis, history["train_loss"], label="train")
ax[0].plot(epochs_axis, history["val_loss"], label="validation")
ax[0].set_title("Perte (plus bas = mieux)"); ax[0].set_xlabel("Époque"); ax[0].legend()
ax[1].plot(epochs_axis, history["train_acc"], label="train")
ax[1].plot(epochs_axis, history["val_acc"], label="validation")
ax[1].set_title("Précision (plus haut = mieux)"); ax[1].set_xlabel("Époque"); ax[1].legend()
for a in ax:
    a.axvline(PHASE1_EPOCHS + 0.5, color="gray", linestyle="--", linewidth=0.8)
plt.tight_layout()
fig.savefig("training_curves.png", dpi=150)
plt.show()

## Étape 6 — Évaluer sur le jeu de test

C'est l'**examen final** : on charge le meilleur modèle et on le teste sur des images qu'il n'a jamais vues.
On calcule les métriques demandées dans l'énoncé :
- **Accuracy** : pourcentage de bonnes réponses.
- **Precision** : quand le modèle annonce une classe, à quelle fréquence a-t-il raison ?
- **Recall** : parmi toutes les vraies images d'une classe, combien le modèle en retrouve-t-il ?
- **F1-score** : un compromis entre precision et recall.
- **Matrice de confusion** : montre quelles classes sont confondues entre elles.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import json

model.load_state_dict(torch.load("best.pt", map_location=device))
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        y_pred.extend(out.argmax(1).cpu().tolist())
        y_true.extend(y.tolist())

acc = accuracy_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, digits=4)
cm = confusion_matrix(y_true, y_pred)

print(f"Accuracy sur le test : {acc:.4f}\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Version simple (types Python purs) pour la sauvegarde
test_metrics = json.loads(json.dumps({
    "accuracy": acc,
    "macro_precision": report["macro avg"]["precision"],
    "macro_recall": report["macro avg"]["recall"],
    "macro_f1": report["macro avg"]["f1-score"],
    "per_class": {c: {"precision": report[c]["precision"], "recall": report[c]["recall"],
                      "f1": report[c]["f1-score"], "support": report[c]["support"]} for c in class_names},
    "confusion_matrix": cm.tolist(),
}, default=float))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
for i in range(len(class_names)):
    for j in range(len(class_names)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("Classe prédite"); ax.set_ylabel("Classe réelle")
ax.set_title(f"Matrice de confusion (test) — accuracy {acc:.3f}")
plt.tight_layout()
fig.savefig("confusion_matrix.png", dpi=150)
plt.show()

## Étape 7 — Sauvegarder le modèle et les résultats

On regroupe dans **un seul fichier** (`neurotrace_model.pt`) tout ce dont l'application aura besoin : les poids du modèle,
les noms des classes, la taille d'image, la version du modèle et les paramètres d'entraînement.
C'est ce qui garantit la **reproductibilité** demandée dans l'énoncé.

Le tout est ensuite compressé dans un `.zip` que tu télécharges sur ton ordinateur.

In [ ]:
import datetime, shutil

MODEL_ID = "resnet18_brain_mri"
MODEL_VERSION = "1.0.0"

checkpoint = {
    "model_id": MODEL_ID,
    "model_version": MODEL_VERSION,
    "arch": "resnet18",
    "class_names": class_names,
    "img_size": IMG_SIZE,
    "mean": MEAN,
    "std": STD,
    "state_dict": {k: v.cpu() for k, v in model.state_dict().items()},
    "test_metrics": test_metrics,
    "training": {
        "dataset": HANDLE,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "phase1_epochs": PHASE1_EPOCHS, "phase1_lr": LR1,
        "phase2_epochs": PHASE2_EPOCHS, "phase2_lr": LR2,
        "best_val_acc": float(best_val_acc),
        "torch_version": torch.__version__,
        "date": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    },
}
torch.save(checkpoint, "neurotrace_model.pt")

with open("metrics.json", "w") as f:
    json.dump({"model_id": MODEL_ID, "model_version": MODEL_VERSION, **test_metrics}, f, indent=2)
with open("class_names.json", "w") as f:
    json.dump(class_names, f)

import os
os.makedirs("neurotrace_model_v1", exist_ok=True)
for name in ["neurotrace_model.pt", "metrics.json", "class_names.json",
             "confusion_matrix.png", "training_curves.png"]:
    shutil.copy(name, "neurotrace_model_v1/")
shutil.make_archive("neurotrace_model_v1", "zip", "neurotrace_model_v1")
print("Archive prête : neurotrace_model_v1.zip")
print(f"Taille du modèle : {os.path.getsize('neurotrace_model.pt')/1e6:.1f} Mo")

In [ ]:
from google.colab import files
files.download("neurotrace_model_v1.zip")

## Étape 8 — Test rapide de la sortie

On recharge le fichier sauvegardé, comme le fera l'application, et on analyse une image de test.
La sortie a **exactement le format du contrat de données** (classe, scores, confiance, version du modèle) :
c'est ce que recevront ensuite la validation et la blockchain.

In [ ]:
ckpt = torch.load("neurotrace_model.pt", map_location="cpu")
m = build_model(len(ckpt["class_names"]), pretrained=False)
m.load_state_dict(ckpt["state_dict"])
m.eval()

path, true_idx = random.choice(test_ds.samples)
img = Image.open(path).convert("RGB")
x = eval_tf(img).unsqueeze(0)
with torch.no_grad():
    probs = torch.softmax(m(x), dim=1)[0]

pred_idx = int(probs.argmax())
result = {
    "case_id": "CASE_TEST",
    "predicted_class": ckpt["class_names"][pred_idx],
    "class_scores": {c: round(float(p), 4) for c, p in zip(ckpt["class_names"], probs)},
    "confidence": round(float(probs[pred_idx]), 4),
    "model_id": ckpt["model_id"],
    "model_version": ckpt["model_version"],
}
print("Classe réelle :", ckpt["class_names"][true_idx])
print(json.dumps(result, indent=2, ensure_ascii=False))
plt.imshow(img); plt.axis("off"); plt.show()

---
**Terminé.** Garde le fichier `neurotrace_model_v1.zip` (il ira dans le dossier `ai/models/` du projet) et reviens dans la
discussion avec Claude en collant l'**accuracy du test** (Étape 6) et la **matrice de confusion**.
Prochaine étape : le prétraitement, la validation et Grad-CAM.